# GAVE2 CMRRWNet V6: Three-Fold Full-Resolution L4 Run

Clean Colab revision V3. Trains Task 2 first, then Task 1, creates a certified base submission, and preserves all prior Drive runs.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

TEAM_ID = "\u68af\u5ea6\u4e0d\u4e0b\u964d\u961f"
RUN_NAME = "gave2_cmrrwnet_v6_3fold"
DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_v6.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
RUN_DIR = DRIVE_BASE / "runs" / RUN_NAME
FOLD_MANIFEST = RUN_DIR / "fold_manifest.json"
OOF_ROOT = RUN_DIR / "predictions" / "oof"
VALIDATION_ROOT = RUN_DIR / "predictions" / "validation"
OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v6_3fold" / "base"
TEAM_ROOT = OUTPUT_ROOT / TEAM_ID
BASE_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v6_3fold_base.zip"
BASE_REPORT = RUN_DIR / "base_submission_report.json"
N_FOLDS = 3
SEED = 77
TASK2_MAX_EPOCHS = 100
TASK1_MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7
EARLY_STOPPING_MIN_DELTA = 1e-4
BASE_CHANNELS = {"task1": 16, "task2": 12}
NUM_REFINEMENTS = 2
AUTO_DISCONNECT = True
RUN_OPTIONAL_REFINER = True
REFINER_EPOCHS = 20
REFINED_OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v6_3fold" / "refined"
REFINED_TEAM_ROOT = REFINED_OUTPUT_ROOT / TEAM_ID
REFINED_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v6_3fold_refined.zip"
REFINED_REPORT = RUN_DIR / "refined_submission_report.json"

def run_module(module, *arguments, capture=False):
    command = [sys.executable, "-m", module, *[str(value) for value in arguments]]
    print("RUN:", " ".join(command), flush=True)
    return subprocess.run(command, cwd=WORK_ROOT, check=True, text=True, capture_output=capture)


## Mount Drive And Extract The V6 Archive


In [ ]:
from google.colab import drive
from pathlib import PurePosixPath

drive.mount("/content/drive")
assert ARCHIVE_PATH.is_file(), f"Missing {ARCHIVE_PATH}"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def normalize_zip_name(name):
    return name.replace("\\", "/").lstrip("/")

required_members = {
    "experiments/gave2_ensemble/train_v6.py",
    "experiments/gave2_ensemble/predict_v6.py",
    "experiments/gave2_ensemble/refiner_v6.py",
    "tests/gave2_ensemble/test_training_v6.py",
}
anchor = "experiments/gave2_ensemble/train_v6.py"

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None, "Archive CRC validation failed"
    entries = archive.infolist()
    normalized_names = [normalize_zip_name(entry.filename) for entry in entries]
    anchor_matches = [name for name in normalized_names if name.endswith(anchor)]
    if len(anchor_matches) != 1:
        raise AssertionError(
            "This is not the new V6 archive. Expected exactly one train_v6.py; "
            f"found {len(anchor_matches)}. First members: {normalized_names[:20]}"
        )
    archive_prefix = anchor_matches[0][:-len(anchor)]
    relative_names = {
        name[len(archive_prefix):]
        for name in normalized_names
        if name.startswith(archive_prefix)
    }
    missing = sorted(required_members - relative_names)
    if missing:
        raise AssertionError(f"V6 archive is missing required files: {missing}")
    if not any(name.startswith("GAVE2_preliminary/") for name in relative_names):
        raise AssertionError(
            "V6 archive has no GAVE2_preliminary data after removing wrapper "
            f"{archive_prefix!r}. First relative members: {sorted(relative_names)[:20]}"
        )

    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    WORK_ROOT.mkdir(parents=True)
    allowed_roots = {"GAVE2_preliminary", "experiments", "tests"}
    work_resolved = WORK_ROOT.resolve()
    for entry, normalized in zip(entries, normalized_names):
        if not normalized.startswith(archive_prefix):
            continue
        relative = normalized[len(archive_prefix):].lstrip("/")
        if not relative:
            continue
        parts = PurePosixPath(relative).parts
        if not parts or parts[0] not in allowed_roots or ".." in parts:
            continue
        destination = WORK_ROOT.joinpath(*parts)
        destination_resolved = destination.resolve()
        if work_resolved != destination_resolved and work_resolved not in destination_resolved.parents:
            raise RuntimeError(f"Unsafe archive member: {entry.filename}")
        if entry.is_dir() or relative.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

assert DATA_ROOT.is_dir(), f"Extraction did not create {DATA_ROOT}"
for required in required_members:
    assert (WORK_ROOT / required).is_file(), f"Extraction missing {required}"
print({
    "archive": str(ARCHIVE_PATH),
    "sha256": sha256_file(ARCHIVE_PATH),
    "members": len(entries),
    "stripped_wrapper": archive_prefix or "<none>",
    "data_root": str(DATA_ROOT),
})


## Install Dependencies And Verify L4

In [ ]:
requirements = WORK_ROOT / "experiments/gave2_ensemble/requirements-gave2-main.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "pytest"],
    check=True,
)

import torch

assert torch.cuda.is_available(), "Select a GPU runtime"
assert torch.cuda.is_bf16_supported(), "The GPU must support BF16"
gpu = torch.cuda.get_device_properties(0)
assert "L4" in gpu.name, f"This notebook is tuned for L4; current GPU is {gpu.name}"
assert gpu.total_memory / 1024**3 >= 20.0

v6_tests = [
    "tests/gave2_ensemble/test_biomarkers_v6.py",
    "tests/gave2_ensemble/test_data_v6.py",
    "tests/gave2_ensemble/test_prediction_v6.py",
    "tests/gave2_ensemble/test_refiner_v6.py",
    "tests/gave2_ensemble/test_training_v6.py",
    "tests/gave2_ensemble/test_train_v6_cli.py",
]
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", *v6_tests, "-q"],
    cwd=WORK_ROOT,
    text=True,
    capture_output=True,
)
print(test_result.stdout, end="")
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr, end="")
if test_result.returncode:
    print(
        "WARNING: Some environment-sensitive V6 unit tests failed in Colab. "
        "The mandatory CUDA model smoke test will decide whether training can proceed.",
        flush=True,
    )

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.cmrrwnet_v6 import create_cmrrwnet_v6

for task, input_channels in (("task1", 4), ("task2", 6)):
    model = create_cmrrwnet_v6(
        task=task,
        base_channels=4,
        num_refinements=1,
        activation_checkpointing=False,
    ).cuda().eval()
    sample = torch.zeros(1, input_channels, 32, 32, device="cuda")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = model(sample)
    final = outputs[-1] if isinstance(outputs, (list, tuple)) else outputs
    assert tuple(final.shape) == (1, 3, 32, 32), (task, tuple(final.shape))
    assert torch.isfinite(final).all(), f"{task} smoke inference produced non-finite values"
    del model, sample, outputs, final
    torch.cuda.empty_cache()

print({
    "torch": torch.__version__,
    "gpu": gpu.name,
    "vram_gib": round(gpu.total_memory / 1024**3, 2),
    "cuda_model_smoke": "passed",
    "pytest_returncode": test_result.returncode,
})


## Build Or Reuse The Immutable Three-Fold Manifest

In [ ]:
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
import numpy as np
from experiments.gave2_ensemble.biomarkers_v2 import locate_optic_disc
from experiments.gave2_ensemble.data import derive_av3_target, list_case_ids, read_png_float
from experiments.gave2_ensemble.data_v6 import build_balanced_three_fold_manifest

RUN_DIR.mkdir(parents=True, exist_ok=True)
case_ids = list_case_ids(DATA_ROOT, split="training")
if not FOLD_MANIFEST.exists():
    records = []
    for case_id in case_ids:
        av = derive_av3_target(read_png_float(DATA_ROOT / "training/av" / f"{case_id}.png", channels=3))
        roi = read_png_float(DATA_ROOT / "training/masks" / f"{case_id}.png", channels=1)[..., 0] > 0.5
        cfp = read_png_float(DATA_ROOT / "training/images" / f"{case_id}.png", channels=3)
        center, _, _ = locate_optic_disc(cfp, av[1], roi)
        records.append({
            "case_id": case_id,
            "artery_prevalence": float(av[0][roi].mean()),
            "vessel_prevalence": float(av[1][roi].mean()),
            "vein_prevalence": float(av[2][roi].mean()),
            "laterality": "L" if center[0] < cfp.shape[1] / 2 else "R",
        })
    manifest = build_balanced_three_fold_manifest(records, seed=SEED)
    FOLD_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
else:
    manifest = json.loads(FOLD_MANIFEST.read_text(encoding="utf-8"))
assert [len(fold["validation"]) for fold in manifest["folds"]] == [17, 17, 16]
assert sorted(case for fold in manifest["folds"] for case in fold["validation"]) == sorted(case_ids)
print({"manifest": str(FOLD_MANIFEST), "validation_sizes": [17, 17, 16]})


## Select The Largest Safe L4 Batch

In [ ]:
from experiments.gave2_ensemble.memory_test_v6 import parse_args as parse_memory_args, run_memory_test

MEMORY_PROFILES = {}
for task in ("task2", "task1"):
    arguments = [
        "--data-root", str(DATA_ROOT), "--fold-manifest", str(FOLD_MANIFEST),
        "--task", task, "--base-channels", str(BASE_CHANNELS[task]),
        "--num-refinements", str(NUM_REFINEMENTS), "--steps", "4",
        "--no-activation-checkpointing",
    ]
    MEMORY_PROFILES[task] = run_memory_test(parse_memory_args(arguments))
(RUN_DIR / "memory_profiles.json").write_text(json.dumps(MEMORY_PROFILES, indent=2), encoding="utf-8")
print(json.dumps(MEMORY_PROFILES, indent=2))


## Train Task 2 Then Task 1

In [ ]:
def train_all_folds(task, epochs):
    profile = MEMORY_PROFILES[task]
    for fold in range(N_FOLDS):
        fold_dir = RUN_DIR / "cmrrwnet_v6" / task / f"fold_{fold}"
        arguments = [
            "--data-root", DATA_ROOT, "--run-dir", RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
            "--task", task, "--fold", fold, "--base-channels", BASE_CHANNELS[task],
            "--num-refinements", NUM_REFINEMENTS, "--batch-size", profile["selected_batch_size"],
            "--workers", 2, "--epochs", epochs, "--amp", "bf16", "--lr", 3e-4,
            "--min-lr", 1e-5, "--weight-decay", 1e-4, "--grad-clip", 1.0,
            "--early-stopping-patience", EARLY_STOPPING_PATIENCE,
            "--early-stopping-min-delta", EARLY_STOPPING_MIN_DELTA,
            "--topology-weight", 0.05, "--seed", SEED, "--no-activation-checkpointing",
        ]
        if (fold_dir / "last.pt").is_file():
            arguments.append("--resume")
        run_module("experiments.gave2_ensemble.train_v6", *arguments)

train_all_folds("task2", TASK2_MAX_EPOCHS)
train_all_folds("task1", TASK1_MAX_EPOCHS)
print("All six folds are trained or exactly resumed.")


## Generate OOF Probabilities And Calibrators

In [ ]:
CALIBRATION = {}
for task in ("task2", "task1"):
    store = OOF_ROOT / task
    run_module(
        "experiments.gave2_ensemble.predict_v6",
        "--data-root", DATA_ROOT, "--run-dir", RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
        "--store-root", store, "--task", task, "--mode", "oof", "--expected-folds", N_FOLDS,
    )
    crossfit = RUN_DIR / "calibration" / f"{task}_crossfit.json"
    run_module(
        "experiments.gave2_ensemble.calibration_v6",
        "--task", task, "--mode", "cross-fit", "--fold-manifest", FOLD_MANIFEST,
        "--prediction-root", store, "--data-root", DATA_ROOT, "--output", crossfit,
    )
    oof_report = RUN_DIR / "calibration" / f"{task}_oof_locked.json"
    oof_report.write_text(json.dumps({"oof_locked": True, "crossfit": str(crossfit)}, indent=2), encoding="utf-8")
    deployment = RUN_DIR / "calibration" / f"{task}_deployment.json"
    run_module(
        "experiments.gave2_ensemble.calibration_v6",
        "--task", task, "--mode", "deployment", "--prediction-root", store,
        "--data-root", DATA_ROOT, "--oof-report", oof_report, "--output", deployment,
    )
    CALIBRATION[task] = deployment
print({task: str(path) for task, path in CALIBRATION.items()})


## Validation Inference And Final Probability PNGs

In [ ]:
for task in ("task2", "task1"):
    store = VALIDATION_ROOT / task
    run_module(
        "experiments.gave2_ensemble.predict_v6",
        "--data-root", DATA_ROOT, "--run-dir", RUN_DIR, "--fold-manifest", FOLD_MANIFEST,
        "--store-root", store, "--task", task, "--mode", "validation", "--expected-folds", N_FOLDS,
    )
    run_module(
        "experiments.gave2_ensemble.predict_v6",
        "--data-root", DATA_ROOT, "--store-root", store, "--task", task, "--mode", "promote",
        "--output-root", OUTPUT_ROOT, "--team-id", TEAM_ID, "--source-split", "validation",
        "--calibration", CALIBRATION[task],
    )
print("Task 1 and Task 2 base outputs complete:", TEAM_ROOT)


## Fit And Generate Task 3

In [ ]:
TASK3_MODEL = RUN_DIR / "task3_target_specific.json"
run_module(
    "experiments.gave2_ensemble.biomarkers_v6", "fit",
    "--data-root", DATA_ROOT, "--prediction-root", OOF_ROOT / "task2", "--output", TASK3_MODEL,
)
run_module(
    "experiments.gave2_ensemble.biomarkers_v6", "predict",
    "--data-root", DATA_ROOT, "--prediction-root", VALIDATION_ROOT / "task2",
    "--model", TASK3_MODEL, "--output-dir", TEAM_ROOT / "Task3", "--split", "validation",
)
assert len(list((TEAM_ROOT / "Task3").glob("*.txt"))) == 50


## Certify Base Submission Before Any Optional Refinement

In [ ]:
from experiments.gave2_ensemble.submission_v6 import certify_submission_atomic, readback_zip

if BASE_ZIP.exists() and BASE_REPORT.exists():
    base_certification = readback_zip(BASE_ZIP, TEAM_ID)
else:
    base_certification = certify_submission_atomic(TEAM_ROOT, DATA_ROOT, BASE_ZIP, BASE_REPORT)
readback = readback_zip(BASE_ZIP, TEAM_ID)
assert readback["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({"ready_to_submit": str(BASE_ZIP), "sha256": readback["sha256"], "counts": readback["counts"]})


## Train And Evaluate The Optional Cross-Fitted Refiner

In [ ]:
REFINER_GATES = {}
REFINER_ROOT = RUN_DIR / "refiners"
REFINED_OOF_ROOT = RUN_DIR / "predictions" / "refined_oof"
REFINED_VALIDATION_ROOT = RUN_DIR / "predictions" / "refined_validation"
REFINED_CALIBRATION = {}

if RUN_OPTIONAL_REFINER:
    for task in ("task2", "task1"):
        try:
            refiner_dir = REFINER_ROOT / task
            run_module(
                "experiments.gave2_ensemble.refiner_v6", "train",
                "--data-root", DATA_ROOT, "--base-store-root", OOF_ROOT / task,
                "--fold-manifest", FOLD_MANIFEST, "--output-dir", refiner_dir,
                "--task", task, "--epochs", REFINER_EPOCHS, "--base-channels", 8,
            )
            for mode, base_root, refined_root in (
                ("oof", OOF_ROOT / task, REFINED_OOF_ROOT / task),
                ("validation", VALIDATION_ROOT / task, REFINED_VALIDATION_ROOT / task),
            ):
                run_module(
                    "experiments.gave2_ensemble.refiner_v6", "predict",
                    "--data-root", DATA_ROOT, "--base-store-root", base_root,
                    "--fold-manifest", FOLD_MANIFEST, "--refiner-dir", refiner_dir,
                    "--output-store-root", refined_root, "--task", task, "--mode", mode,
                )
            refined_crossfit = RUN_DIR / "calibration" / f"{task}_refined_crossfit.json"
            run_module(
                "experiments.gave2_ensemble.calibration_v6",
                "--task", task, "--mode", "cross-fit", "--fold-manifest", FOLD_MANIFEST,
                "--prediction-root", REFINED_OOF_ROOT / task, "--data-root", DATA_ROOT,
                "--output", refined_crossfit,
            )
            refined_locked = RUN_DIR / "calibration" / f"{task}_refined_oof_locked.json"
            refined_locked.write_text(json.dumps({"oof_locked": True, "crossfit": str(refined_crossfit)}, indent=2), encoding="utf-8")
            refined_deployment = RUN_DIR / "calibration" / f"{task}_refined_deployment.json"
            run_module(
                "experiments.gave2_ensemble.calibration_v6",
                "--task", task, "--mode", "deployment", "--prediction-root", REFINED_OOF_ROOT / task,
                "--data-root", DATA_ROOT, "--oof-report", refined_locked, "--output", refined_deployment,
            )
            comparison = RUN_DIR / "refiners" / f"{task}_gate.json"
            run_module(
                "experiments.gave2_ensemble.refiner_v6", "compare",
                "--data-root", DATA_ROOT, "--base-store-root", OOF_ROOT / task,
                "--refined-store-root", REFINED_OOF_ROOT / task,
                "--base-calibration", RUN_DIR / "calibration" / f"{task}_crossfit.json",
                "--refined-calibration", refined_crossfit, "--task", task, "--output", comparison,
            )
            comparison_report = json.loads(comparison.read_text(encoding="utf-8"))
            REFINER_GATES[task] = comparison_report["gate"]
            REFINED_CALIBRATION[task] = refined_deployment
        except Exception as exc:
            REFINER_GATES[task] = {"accepted": False, "error": repr(exc)}
            print(f"Optional {task} refiner failed; preserving base output: {exc}", flush=True)
else:
    REFINER_GATES = {task: {"accepted": False, "reason": "disabled"} for task in ("task2", "task1")}

(RUN_DIR / "refiner_gates.json").write_text(json.dumps(REFINER_GATES, indent=2), encoding="utf-8")
print(json.dumps(REFINER_GATES, indent=2))


## Build And Certify The Refined Candidate

In [ ]:
if not (REFINED_ZIP.exists() and REFINED_REPORT.exists()):
    REFINED_TEAM_ROOT.mkdir(parents=True, exist_ok=True)
    for task_name in ("Task1", "Task2", "Task3"):
        source = TEAM_ROOT / task_name
        destination = REFINED_TEAM_ROOT / task_name
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)

    for task in ("task2", "task1"):
        if REFINER_GATES.get(task, {}).get("accepted", False):
            run_module(
                "experiments.gave2_ensemble.predict_v6",
                "--data-root", DATA_ROOT, "--store-root", REFINED_VALIDATION_ROOT / task,
                "--task", task, "--mode", "promote", "--output-root", REFINED_OUTPUT_ROOT,
                "--team-id", TEAM_ID, "--source-split", "validation",
                "--calibration", REFINED_CALIBRATION[task],
            )

    if REFINER_GATES.get("task2", {}).get("accepted", False):
        refined_task3_model = RUN_DIR / "task3_refined_target_specific.json"
        run_module(
            "experiments.gave2_ensemble.biomarkers_v6", "fit",
            "--data-root", DATA_ROOT, "--prediction-root", REFINED_OOF_ROOT / "task2",
            "--output", refined_task3_model,
        )
        if (REFINED_TEAM_ROOT / "Task3").exists():
            shutil.rmtree(REFINED_TEAM_ROOT / "Task3")
        run_module(
            "experiments.gave2_ensemble.biomarkers_v6", "predict",
            "--data-root", DATA_ROOT, "--prediction-root", REFINED_VALIDATION_ROOT / "task2",
            "--model", refined_task3_model, "--output-dir", REFINED_TEAM_ROOT / "Task3",
            "--split", "validation",
        )
    refined_certification = certify_submission_atomic(
        REFINED_TEAM_ROOT, DATA_ROOT, REFINED_ZIP, REFINED_REPORT
    )
else:
    refined_certification = readback_zip(REFINED_ZIP, TEAM_ID)

refined_readback = readback_zip(REFINED_ZIP, TEAM_ID)
assert refined_readback["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print({"refined_candidate": str(REFINED_ZIP), "sha256": refined_readback["sha256"], "gates": REFINER_GATES})


## Release Runtime After Verified ZIP

In [ ]:
assert BASE_ZIP.is_file() and BASE_REPORT.is_file()
assert REFINED_ZIP.is_file() and REFINED_REPORT.is_file()
assert readback_zip(BASE_ZIP, TEAM_ID)["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
assert readback_zip(REFINED_ZIP, TEAM_ID)["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
if AUTO_DISCONNECT:
    print("Both submission candidates verified. Disconnecting in 10 seconds.", flush=True)
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Both submission candidates verified. AUTO_DISCONNECT is disabled.")
